# haluviz — fit a Jacobian lens for Qwen-1.8B-Chat on a Colab GPU

Runtime → Change runtime type → **T4 GPU** before running.
Run the cells top to bottom. The lens is written to your Google Drive so a disconnect doesn't lose it.

In [ ]:
# 1. Drive (for the output) + upload haluviz_colab.zip from the project folder when prompted
from google.colab import drive, files
drive.mount('/content/drive')
import os; os.makedirs('/content/drive/MyDrive/haluviz', exist_ok=True)
up = files.upload()
!rm -rf /content/haluviz && mkdir -p /content/haluviz && unzip -q -o haluviz_colab.zip -d /content/haluviz
%cd /content/haluviz
!pip install -q transformers gguf safetensors datasets huggingface_hub
import torch; print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Convert the GGUF into a native checkpoint (~3 min: downloads 3.7 GB, writes 3.6 GB)
!python scripts/convert_gguf_to_hf.py --out /content/model

In [ ]:
# 3. Fit the lens. ~1 min per prompt on a T4; 30 prompts is a good lens, 10 is already usable.
#    Checkpointed: if Colab disconnects, re-run this cell and it resumes.
#    A usable partial lens is written to Drive after every prompt.
OUT = '/content/drive/MyDrive/haluviz/qwen-1_8b-chat_jlens_wikitext.pt'
!python scripts/fit_lens.py --model-id /content/model --dtype float32 \
    --n-prompts 30 --dim-batch 8 --max-seq-len 128 --out {OUT}

In [ ]:
# 4. Sanity check: mid-layer readouts under the J-lens vs the logit lens for one prompt
import sys; sys.path.insert(0, '/content/haluviz')
from jspace_viz.model import load_model
from jspace_viz.lens import JacobianLens
from jspace_viz.analysis import read_grid
m = load_model('/content/model', dtype='float16'); lens = JacobianLens.load(OUT)
print(lens)
for mode in ('logit', 'jlens'):
    r = read_grid(m, lens, 'What is the capital city of Palau? Answer in one sentence.', mode=mode, chat=True, max_new_tokens=12, top_k=3)
    col = r['prompt_len'] - 1  # the position that predicts the first answer word
    print(f"\n{mode}: answer={r['continuation']!r}")
    for row in r['grid'][::3]:
        print(f"  L{row['layer']:2d}: {[r['vocab'][i] for i in row['top_ids'][col]]}")

print('\nDone. The lens is in Drive: MyDrive/haluviz/. Copy it to lenses/ in the project.')